In [ ]:
!pip install -q pandas plotly

In [ ]:
#librerias para trabajar con datos
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Carga del dataset directo desde Hugging Face usando Pandas
url_hf = "hf://datasets/aptologia/microestudios-barrios-medellin-2026/indice.csv"

try:
  df_inmuebles= pd.read_csv(url_hf)
  print("cargo datos bien")
except Exception as e:
  print("Error al cargar el conjunto de datos")

#probamos inicialmente
df_inmuebles.head()

In [ ]:
print("Columnas disponibles:" , df_inmuebles.columns.tolist())

df_clean = df_inmuebles.dropna(subset=['barrio', 'n_avisos', 'mediana_cop', 'mediana_cop_m2','operacion','tipo' ]).copy()
df_clean['n_avisos'] = pd.to_numeric(df_clean['n_avisos'], errors='coerce')
df_clean['mediana_cop'] = pd.to_numeric(df_clean['mediana_cop'], errors='coerce')
df_clean['mediana_cop_m2'] = pd.to_numeric(df_clean['mediana_cop_m2'], errors='coerce')

In [ ]:
# Resumen agrupado por Barrio y Tipo de Operación
resumen_barrios = df_clean.groupby(['barrio', 'operacion']).agg(
    precio_promedio=('mediana_cop', 'mean'),
    precio_mediana=('mediana_cop', 'median'),
    cantidad_ofertas=('mediana_cop', 'count')
).reset_index()

# Visualización 1: Barrios con mayor volumen de oferta (Objetivo 3)
top_barrios_oferta = df_clean['barrio'].value_counts().head(15).reset_index()
top_barrios_oferta.columns = ['barrio', 'n_avisos']

fig_oferta = px.bar(
    top_barrios_oferta,
    x='n_avisos',
    y='barrio',
    orientation='h',
    title=' Top 15 Barrios con Mayor Oferta Inmobiliaria en Medellín',
    labels={'n_avisos': 'Número de Propiedades Disponibles', 'barrio': 'Barrio'},
    color='n_avisos',
    color_continuous_scale='Viridis'
)
#fig_oferta.update_layout(yaxis={'categoryorder': 'n_avisos'})
fig_oferta.show()

# Visualización 2: Promedio de costo por Barrio (Objetivo 1)
fig_costo = px.bar(
    resumen_barrios.sort_values(by='precio_promedio', ascending=False).head(20),
    x='barrio',
    y='precio_promedio',
    color='operacion',
    barmode='group',
    title='💰 Promedio de Costo por Barrio y Tipo de Operación (Venta / Arriendo)',
    labels={'precio_promedio': 'Precio Promedio ($)', 'barrio': 'Barrio'}
)
fig_costo.show()

In [ ]:
def perfilar_barrio(barrio_consulta: str):
    """
    Función para filtrar y consultar la disponibilidad y costos en un barrio específico.
    """
    df_barrio = df_clean[df_clean['barrio'].str.contains(barrio_consulta, case=False, na=False)]

    if df_barrio.empty:
        print(f"⚠️ No se encontraron registros para el barrio: '{barrio_consulta}'")
        return

    print("=" * 60)
    print(f"📌 INFORME DE PERFILAMIENTO: BARRIO {barrio_consulta.upper()}")
    print("=" * 60)

    resumen = df_barrio.groupby('tipo_operacion').agg(
        Opciones_Disponibles=('mediana_cop', 'count'),
        Precio_Promedio=('mediana_cop', 'mean'),
        Precio_Minimo=('mediana_cop', 'min'),
        Precio_Maximo=('mediana_cop', 'max')
    ).reset_index()

    # Formatear salida monetaria
    resumen['Precio_Promedio'] = resumen['Precio_Promedio'].apply(lambda x: f"${x:,.2f}")
    resumen['Precio_Minimo'] = resumen['Precio_Minimo'].apply(lambda x: f"${x:,.2f}")
    resumen['Precio_Maximo'] = resumen['Precio_Maximo'].apply(lambda x: f"${x:,.2f}")

    display(resumen)

# Ejemplo de uso de la herramienta de filtrado:
barrio_interes = "El Poblado"  # Cambiar por el barrio deseado
perfilar_barrio(barrio_interes)

In [ ]:
#generacion del archivo:
from pathlib import Path

# 1. Definir la ruta del archivo de salida
nombre_archivo = "resumen_dataframe.txt"
ruta_salida = Path(nombre_archivo)

# 2. Construir el contenido del reporte paso a paso
lineas_reporte = []

lineas_reporte.append("=" * 70)
lineas_reporte.append("INFORME DE PERFILAMIENTO Y PRESUPUESTO INMOBILIARIO - MEDELLÍN")
lineas_reporte.append("=" * 70)
lineas_reporte.append(f"Fuente de datos: hf://datasets/aptologia/microestudios-barrios-medellin-2026/indice.csv")
lineas_reporte.append(f"Total de registros analizados: {len(df_clean)}")
lineas_reporte.append("\n")

# --- SECCIÓN 1: Top Barrios con Mayor Oferta (Objetivo 3) ---
lineas_reporte.append("-" * 70)
lineas_reporte.append("1. BARRIOS CON MAYOR VOLUMEN DE OFERTA (TOP 15)")
lineas_reporte.append("-" * 70)

top_oferta = df_clean['barrio'].value_counts().head(15)
for i, (barrio, n_avisos) in enumerate(top_oferta.items(), 1):
    lineas_reporte.append(f"{i:2d}. {barrio:<35} : {n_avisos} ofertas")

lineas_reporte.append("\n")

# --- SECCIÓN 2: Costo Promedio y Disponibilidad por Barrio (Objetivos 1 y 2) ---
lineas_reporte.append("-" * 70)
lineas_reporte.append("2. PROMEDIO DE COSTO Y CANTIDAD DE NEGOCIOS POR BARRIO Y TIPO DE OPERACIÓN")
lineas_reporte.append("-" * 70)

resumen_completo = df_clean.groupby(['barrio', 'operacion']).agg(
    ofertas=('mediana_cop', 'count'),
    promedio=('mediana_cop', 'mean'),
    minimo=('mediana_cop', 'min'),
    maximo=('mediana_cop', 'max')
).reset_index()

# Ordenar por mayor cantidad de ofertas
resumen_completo = resumen_completo.sort_values(by=['ofertas', 'promedio'], ascending=[False, False])

lineas_reporte.append(f"{'BARRIO':<30} | {'OPERACIÓN':<10} | {'OFERTAS':<8} | {'PROMEDIO ($)':<18} | {'RANGO ($)':<25}")
lineas_reporte.append("-" * 100)

for _, fila in resumen_completo.iterrows():
    barrio = str(fila['barrio'])[:28]
    operacion = str(fila['operacion'])
    ofertas = fila['n_avisos']
    prom = f"${fila['promedio']:,.2f}"
    rango = f"${fila['minimo']:,.0f} - ${fila['maximo']:,.0f}"
    
    lineas_reporte.append(f"{barrio:<30} | {operacion:<10} | {ofertas:<8} | {prom:<18} | {rango:<25}")

lineas_reporte.append("\n" + "=" * 70)
lineas_reporte.append("FIN DEL INFORME")
lineas_reporte.append("=" * 70)

# 3. Unir todas las líneas en un solo texto
contenido_txt = "\n".join(lineas_reporte)

# 4. Escribir el archivo en disco
ruta_salida.write_text(contenido_txt, encoding="utf-8")

print(f"✅ Archivo guardado exitosamente como: {ruta_salida.resolve()}\n")

# 5. (Opcional) Descargar el archivo directamente en el navegador desde Colab
try:
    from google.colab import files
    files.download(nombre_archivo)
    print("📥 Descarga del archivo iniciada automáticamente.")
except ImportError:
    print("ℹ️ Ejecutando fuera de Google Colab. El archivo está disponible en la carpeta local.")